# Facial Recognition Misidentification: When the Data Itself Is the Bottleneck

In this notebook I look at wrongful facial recognition matches as a fourth, distinct mechanism behind an outcome gap, closing out this run of AI ethics case studies.

The Tokyo Medical University notebook was a deliberate manual adjustment. The COMPAS notebook was an honest score meeting differing base rates. The Amazon notebook was a neutral feature acting as an accidental proxy. This one is different again: a group that a system saw far less of during training can end up with a genuinely worse feature representation, one that does not separate different people in that group as clearly, and no amount of threshold tuning fixes that on its own.

In this notebook, I will:

- Summarize a real wrongful arrest case and what a major accuracy study found
- Simulate a matching system where one group is underrepresented in training
- Measure the false match rate gap that shows up as a result
- Try the fix that worked in earlier notebooks, a per-group threshold, and watch it fail to close the gap this time
- Find the fix that actually works, and see why it is a very different kind of fix from the ones before it

## 1. Background: What Was Reported

In January 2020, Robert Williams was arrested in Detroit after a facial recognition search matched a photo from a surveillance video to his driver's license photo. The match was wrong. He was held for roughly 30 hours before the case against him fell apart, and his case, brought with the ACLU, is widely reported as the first publicly known wrongful arrest in the United States tied to a facial recognition match.

Separately, in 2019 the National Institute of Standards and Technology (NIST) published a large accuracy study of facial recognition algorithms from many vendors. It found that a majority of the algorithms tested showed higher false positive rates, incorrectly matching two different people, for some demographic groups than others, though NIST was clear that the size of this gap varied enormously by algorithm, and a handful of the algorithms tested showed close to no such gap at all. The study did not settle on one single cause, but pointed to training data composition as one contributing factor among several.

## 2. Why This Is a Different Mechanism

A face recognition system does not compare raw photos, it compares learned numerical features extracted from them, positions in a space the model built during training. If a group was a small share of that training data, the model had far fewer examples to learn from about what actually varies between different people in that group, and what stays the same across different photos of the same person.

The plausible result is not necessarily noisier photos, it is a feature space that does not spread different people in that group out as clearly. Two different people can end up mapped closer together than two different people from a group the model saw plenty of, which raises the odds of exactly the kind of error that led to Robert Williams's arrest, a system reporting a confident match between two different people.

## 3. A Note on the Simulation Below

This notebook does not use real photos, real embeddings, or a real trained model. I represent each simulated person's identity as a single number, and each "photo" of them as that number plus some random noise, standing in for the variation between different photos of the same person. It is a large simplification of a high-dimensional deep learning system, built to make one specific mechanism, poor separation between people in an underrepresented group, easy to see and measure.

## 4. Simulating a Well-Represented and an Underrepresented Group

Group A gets many simulated identities spread across the full range the feature space can represent. Group B gets far fewer identities, and I compress them into a narrower slice of that same range, standing in for a model that never learned to spread this group's faces out as widely, because it saw so few of them during training. Both groups get the exact same amount of photo-to-photo noise, so any gap that shows up is coming from the spread of identities, not from noisier pictures.

In [ ]:
import random

random.seed(3)

NOISE = 5


def make_identities(count, low, high):
    """Returns a list of identity codes spread across a range."""
    return [random.uniform(low, high) for _ in range(count)]


def photo_feature(identity_code, noise_sigma=NOISE):
    """Returns a noisy feature reading for one photo of an identity."""
    return identity_code + random.gauss(0, noise_sigma)


identities_a = make_identities(180, low=0, high=100)
identities_b = make_identities(20, low=0, high=30)

## 5. Building Same-Person and Different-Person Pairs

To evaluate a matching rule, I need two kinds of pairs: two photos of the same identity, and two photos of two different identities. I write one function that produces both.

In [ ]:
def make_pairs(identities, n_same, n_diff):
    """Returns absolute feature differences for same-person and different-person pairs."""
    same_pairs = []
    for _ in range(n_same):
        identity = random.choice(identities)
        difference = abs(photo_feature(identity) - photo_feature(identity))
        same_pairs.append(difference)

    diff_pairs = []
    for _ in range(n_diff):
        identity_1, identity_2 = random.sample(identities, 2)
        difference = abs(photo_feature(identity_1) - photo_feature(identity_2))
        diff_pairs.append(difference)

    return same_pairs, diff_pairs

## 6. Generating Training Pairs for Both Groups

Group A contributes far more training pairs than group B, mirroring the real situation: a system trained mostly on one group's data has that much more evidence from it when a threshold gets chosen.

In [ ]:
same_a, diff_a = make_pairs(identities_a, n_same=900, n_diff=900)
same_b, diff_b = make_pairs(identities_b, n_same=100, n_diff=100)

## 7. Estimating a Single Shared Matching Threshold

Now I pick one threshold, the same for everyone, the way a single deployed system typically would. I pool the same-person pairs from both groups and set the threshold at the 95th percentile, loose enough to accept the vast majority of genuine same-person pairs. Because group A supplied nine times as many pairs as group B, this threshold is shaped mostly by group A's data.

In [ ]:
def percentile(values, fraction):
    """Returns the value at a given percentile of a list."""
    ordered = sorted(values)
    index = int(fraction * len(ordered))
    return ordered[min(index, len(ordered) - 1)]


pooled_same = same_a + same_b
shared_threshold = percentile(pooled_same, 0.95)
print("Shared threshold:", round(shared_threshold, 2))

## 8. Measuring the False Match Rate Gap

I generate a brand-new batch of test pairs for each group and check two things at the shared threshold: the false match rate, how often two different people get called the same person, and the false non-match rate, how often two photos of the same person get called different people.

In [ ]:
def false_match_rate(diff_pairs, threshold):
    """Returns the fraction of different-person pairs wrongly called a match."""
    wrong = [d for d in diff_pairs if d < threshold]
    return len(wrong) / len(diff_pairs)


def false_non_match_rate(same_pairs, threshold):
    """Returns the fraction of same-person pairs wrongly called different."""
    wrong = [d for d in same_pairs if d >= threshold]
    return len(wrong) / len(same_pairs)


test_same_a, test_diff_a = make_pairs(identities_a, n_same=1000, n_diff=1000)
test_same_b, test_diff_b = make_pairs(identities_b, n_same=1000, n_diff=1000)

print("False match rate, group A:", round(false_match_rate(test_diff_a, shared_threshold), 3))
print("False match rate, group B:", round(false_match_rate(test_diff_b, shared_threshold), 3))
print("False non-match rate, group A:", round(false_non_match_rate(test_same_a, shared_threshold), 3))
print("False non-match rate, group B:", round(false_non_match_rate(test_same_b, shared_threshold), 3))

## 9. Trying a Per-Group Threshold

In the COMPAS notebook, a per-group threshold changed the outcome, at the cost of breaking calibration. In the Amazon notebook, removing the one proxy feature fixed the gap outright. I try the natural next thing here: calibrate the threshold separately for each group, using only that group's own same-person pairs, and see whether the same kind of fix works this time.

In [ ]:
threshold_a = percentile(same_a, 0.95)
threshold_b = percentile(same_b, 0.95)

print("Group A's own threshold:", round(threshold_a, 2))
print("Group B's own threshold:", round(threshold_b, 2))

print("False match rate, group A, own threshold:", round(false_match_rate(test_diff_a, threshold_a), 3))
print("False match rate, group B, own threshold:", round(false_match_rate(test_diff_b, threshold_b), 3))